⚛️ Quantum Process Tomography (QPT)\\
Quantum Process Tomography is an experimental procedure used to completely characterize an unknown quantum operation (or quantum channel), $\mathcal{E}$, which governs the evolution of quantum states. It provides the most complete diagnostic tool for validating quantum hardware and assessing the quality of quantum gates.
The goal is to determine the process matrix ($\chi$), which fully describes the map $\mathcal{E}$. The map $\mathcal{E}$ transforms an input state $\rho_{\text{in}}$ into an output state $\rho_{\text{out}}$ according to the Kraus representation:
$$\rho_{\text{out}} = \mathcal{E}(\rho_{\text{in}}) = \sum_{m,n} \chi_{mn} \hat{A}_m \rho_{\text{in}} \hat{A}^\dagger_n$$
where $\{\hat{A}_m\}$ is a set of basis operators (often the Pauli basis $\{\hat{I}, \hat{\sigma}_x, \hat{\sigma}_y, \hat{\sigma}_z\}$ for a single qubit), and $\chi$ is the process matrix. The matrix $\chi$ is a $d^2 \times d^2$ Hermitian matrix (where $d$ is the dimension of the system's Hilbert space) that fully captures both unitary (coherent) dynamics and non-unitary (decoherent or noisy) effects.

🔬 Experimental Implementation Steps\\
QPT is essentially a two-part tomography process: preparation tomography and measurement tomography. It involves a systematic set of input states and output measurements.
Step 1: Prepare a Tomographically Complete Set of Input States
A quantum process must be probed using a set of input states $\{\rho_j\}$ that are tomographically complete. For a $d$-dimensional system, this set typically requires $d^2$ linearly independent states to fully characterize the process.
For a single qubit ($d=2$): $d^2=4$ input states are needed. A common choice is the computational basis states and their superpositions, such as the six pure states corresponding to the eigenstates of the Pauli operators:
$\hat{\sigma}_z$ eigenstates: $|0\rangle$, $|1\rangle$
$\hat{\sigma}_x$ eigenstates: $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, $|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$
$\hat{\sigma}_y$ eigenstates: $|+i\rangle = \frac{1}{\sqrt{2}}(|0\rangle + i|1\rangle)$, $|-i\rangle = \frac{1}{\sqrt{2}}(|0\rangle - i|1\rangle)$
Procedure: Prepare the quantum system (e.g., a qubit) in the initial state $\rho_j$.
Step 2: Apply the Unknown Process ($\mathcal{E}$)
The core of the experiment involves applying the unknown quantum operation, $\mathcal{E}$ (which you refer to as the "MCED process"), to each of the prepared input states $\rho_j$.
Output State: Each input state $\rho_j$ is transformed into a corresponding output state: $\rho'_j = \mathcal{E}(\rho_j)$.
Repetition: This step is repeated many times for each $\rho_j$ to gather sufficient statistical data in the following measurement step.
Step 3: Perform Quantum State Tomography (QST) on the Output State
For each output state $\rho'_j$, a second tomographically complete set of measurements must be performed to reconstruct the state itself. This is Quantum State Tomography (QST).
Measurement Basis: For a single qubit, this typically involves measurements in three different bases: the $Z$, $X$, and $Y$ bases (corresponding to measuring $\hat{\sigma}_z$, $\hat{\sigma}_x$, and $\hat{\sigma}_y$).
Data Collection: For a given input state $\rho_j$, the system is subjected to the process $\mathcal{E}$, and then the probability of measuring each outcome in the measurement basis is recorded.
The total number of required experimental settings (Input State $\times$ Measurement Setting) scales as $d^4$ for a $d$-dimensional system. For a single qubit ($d=2$), this is $2^4 = 16$ settings.
Step 4: Data Processing and Matrix Reconstruction
The final step is to use the collected experimental data (the measured expectation values of the output states for all input/measurement combinations) to solve for the unknown $\chi$ matrix.
Linear Inversion: The measured output expectation values are related to the process matrix $\chi$ by a set of linear equations. The standard method involves solving this system to find an initial estimate for $\chi$.
Constraint Optimization: The linear inversion method often produces an unphysical process matrix that does not satisfy the constraints of a valid quantum channel (it may not be completely positive and trace-preserving, or CPTP). To correct this, advanced techniques like Maximum Likelihood Estimation (MLE) are used.
MLE finds the physical, CPTP process matrix $\chi$ that is most likely to have produced the observed experimental data.
Result Interpretation: The resulting $\chi$ matrix is the complete characterization of your "MCED process." Analyzing $\chi$ allows you to calculate critical performance metrics, such as:
Process Fidelity ($F_p$): How close the experimental process $\mathcal{E}$ is to the ideal target process $\mathcal{E}_{\text{ideal}}$.
Purity, Entangling Capability, and Choi state properties.

1. The Realistic Dataset structure
In a superconducting qubit experiment (e.g., using IBM Qiskit or similar), you do not get probabilities directly; you get shot counts (bitstrings).
For a 1-qubit Process Tomography, you run 12 specific circuits.
4 Input States: $\{|0\rangle, |1\rangle, |+\rangle, |+i\rangle\}$
3 Measurement Bases: $\{X, Y, Z\}$ for each input.
Shots per circuit: Typically $1000 - 8000$ shots.
Sample Data Table (Snippet for Input $|0\rangle$):
| Input State | Measurement Basis | Counts {'0': n0, '1': n1} |
| :--- | :--- | :--- |
| $|0\rangle$ | $Z$ (Standard) | {'0': 980, '1': 20} |
| $|0\rangle$ | $X$ (via $R_y(-\pi/2)$) | {'0': 510, '1': 490} |
| $|0\rangle$ | $Y$ (via $R_x(\pi/2)$) | {'0': 485, '1': 515} |
Note: In this realistic data, even if the process is "Identity", the $|0\rangle \to Z$ measurement isn't perfect (1000:0) due to readout error ($~2\%$) and thermal noise.

2. Step 1: Readout Error Mitigation (Pre-processing)
You pre-measure a Confusion Matrix $M$ by preparing $|0\rangle$ and $|1\rangle$ and measuring them immediately:
$$M = \begin{pmatrix} P(0|0) & P(0|1) \\ P(1|0) & P(1|1) \end{pmatrix} \approx \begin{pmatrix} 0.99 & 0.03 \\ 0.01 & 0.97 \end{pmatrix}$$
$P(0|1) = 0.03$ implies a $3\%$ error where a $|1\rangle$ looks like a $0$.
You apply the inverse $M^{-1}$ to your raw counts vector $\vec{C}_{\text{raw}} = [n_0, n_1]^T$ to get the "mitigated" counts $\vec{C}_{\text{mit}}$.
$$\vec{C}_{\text{mit}} = M^{-1} \vec{C}_{\text{raw}}$$

3. Step 2: Convert to Expectation Values (Stokes Vectors)
For every input state $\rho_j$, you effectively perform Quantum State Tomography (QST) on the output. You convert the mitigated counts into expectation values for the Pauli operators.
For input $\rho_j$ and measurement basis $M \in \{X, Y, Z\}$:
$$\langle \sigma_M \rangle_j = \frac{n_0^{\text{mit}} - n_1^{\text{mit}}}{n_0^{\text{mit}} + n_1^{\text{mit}}}$$

4. Step 3: Matrix Reconstruction (The Math)
The goal is to find the Process Matrix $\chi$ (a $4\times4$ matrix) that satisfies:
$$\mathcal{E}(\rho) = \sum_{m,n=0}^{3} \chi_{mn} \sigma_m \rho \sigma_n$$
(where $\sigma_0 = I$).
Method B: Maximum Likelihood Estimation (MLE) / Convex Optimization
This is the standard for realistic datasets.
Objective: Find the matrix $\chi$ that is Completely Positive Trace Preserving (CPTP) and minimizes the distance between the predicted data and observed data.
$$\text{Minimize: } f(\chi) = \sum_{j, M} \left( \langle \sigma_M \rangle_j^{\text{observed}} - \text{Tr}[\sigma_M \mathcal{E}_\chi(\rho_j)] \right)^2$$
Constraints:
Hermiticity: $\chi = \chi^\dagger$
Positivity: $\chi \ge 0$ (All eigenvalues $\ge 0$).
Trace Preservation: $\sum_{mn} \chi_{mn} \sigma_n \sigma_m = I$.

In [ ]:
The Python Implementation
Python
import numpy as np
import cvxpy as cp

# --- 1. PRELIMINARIES: Define Basis and Helper Functions ---

# Define Single Qubit Pauli Basis: {I, X, Y, Z}
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
Paulis = [I, X, Y, Z]

def get_rho(state_index):
    """Returns the density matrix for the 4 calibration input states."""
    # 0: |0>, 1: |1>, 2: |+>, 3: |+i>
    if state_index == 0: return np.array([[1,0],[0,0]], dtype=complex)
    if state_index == 1: return np.array([[0,0],[0,1]], dtype=complex)
    if state_index == 2: return 0.5 * np.array([[1,1],[1,1]], dtype=complex)
    if state_index == 3: return 0.5 * np.array([[1,-1j],[1j,1]], dtype=complex)
    raise ValueError("Invalid state index")

# --- 2. GENERATE "REALISTIC" DATASET --- 
# The "True" Process Matrix for Identity (Ideal)
chi_true = np.zeros((4, 4), dtype=complex)
chi_true[0, 0] = 1.0 

experiment_data = []

for i_prep in range(4):
    rho_in = get_rho(i_prep)
    
    # Apply the process: E(rho) = Sum( chi_mn * Pm * rho * Pn_dag )
    rho_out_ideal = np.zeros((2,2), dtype=complex)
    for m in range(4):
        for n in range(4):
            rho_out_ideal += chi_true[m, n] * Paulis[m] @ rho_in @ Paulis[n].conj().T
            
    for j_meas, P_meas in enumerate([X, Y, Z]):
        expect_val_ideal = np.real(np.trace(P_meas @ rho_out_ideal))
        
        # ADD NOISE: Simulate finite sampling/shot noise
        noise = np.random.normal(0, 0.05) 
        measured_val = expect_val_ideal + noise
        
        experiment_data.append((i_prep, j_meas, measured_val))


# --- 3. RECONSTRUCTION VIA CVXPY ---
chi_var = cp.Variable((4, 4), hermitian=True)

# Build the Cost Function (L2 Norm / Least Squares)
cost = 0
constraints = []

for (i_prep, j_meas, measured_val) in experiment_data:
    rho_in = get_rho(i_prep)
    P_meas = [X, Y, Z][j_meas]
    
    # Calculate the coefficient matrix for this specific data point
    coeffs = np.zeros((4, 4), dtype=complex)
    for m in range(4):
        for n in range(4):
            term = P_meas @ Paulis[m] @ rho_in @ Paulis[n].conj().T
            coeffs[m, n] = np.trace(term)
            
    # The predicted value is the Frobenius inner product of chi_var and coeffs
    prediction = cp.real(cp.trace(coeffs.T @ chi_var))
    
    # Add square error to cost
    cost += cp.square(prediction - measured_val)

# --- 4. ADD PHYSICAL CONSTRAINTS ---

# Constraint A: Chi must be Positive Semidefinite (CP)
constraints.append(chi_var >> 0)

# Constraint B: Trace Preserving (TP)
tp_sum = 0
for m in range(4):
    for n in range(4):
        tp_sum += chi_var[m, n] * Paulis[n].conj().T @ Paulis[m]

constraints.append(tp_sum == I)

# --- 5. SOLVE ---
prob = cp.Problem(cp.Minimize(cost), constraints)
prob.solve()

# --- 6. RESULTS ---
chi_reconstructed = chi_var.value

# Calculate Fidelity with Ideal Identity
fidelity = np.real(np.trace(chi_true @ chi_reconstructed))


1. The Diagonal: Stochastic Noise (Incoherent)
The diagonal elements ($\chi_{mm}$) represent classical probabilities of discrete errors occurring. These are errors that destroy quantum information (decoherence).
$\chi_{00}$ (The "Identity" term): This should be close to 1.0. It represents the "good" part of the process.
$\chi_{11}, \chi_{22}, \chi_{33}$ (The Pauli terms): These represent Depolarizing Noise.
$\chi_{33}$ (the $Z$ position) indicates a Phase Flip ($Z$ error), often due to $T_2$ dephasing.
Diagnosis: If you have large diagonal elements (other than $\chi_{00}$), your qubit is interacting with the environment (incoherent error).

2. The Off-Diagonals: Coherent Errors (Calibration)
Off-diagonal terms (specifically between Identity and a Pauli) indicate that your gate is systematically wrong in a unitary way (coherent error), usually due to miscalibrated control pulses.
Case A: Over-Rotation (Amplitude Error)
A slight rotation around the X-axis ($U = e^{-i \epsilon X / 2}$).
The Signature:
You will see purely imaginary values at $\chi_{01}$ ($\chi_{IX}$) and $\chi_{10}$ ($\chi_{XI}$).
Action: Your microwave pulse amplitude is likely too high. Scale down the pulse amplitude.
Case B: Phase Drift / Detuning (Frequency Error)
The microwave drive frequency is slightly off-resonance, causing a slow rotation around the Z-axis ($U = e^{-i \epsilon Z / 2}$).
The Signature:
You will see purely imaginary values at $\chi_{03}$ ($\chi_{IZ}$) and $\chi_{30}$ ($\chi_{ZI}$).
Action: Adjust the frequency of your local oscillator (LO) or update the qubit frequency definition.

3. Summary Diagnostic Table
When characterizing an intended Identity or Idle process:
Observation in χ
Interpretation
Physical Cause
Fix
High $\chi_{ZZ}$ (Real)
Phase Flip Noise
$T_2$ Dephasing
Use Echo pulses / better shielding
Imaginary $\chi_{IX}, \chi_{XI}$
Coherent X-Rotation
Over-rotation (Rabi error)
Recalibrate Pulse Amplitude
Imaginary $\chi_{IZ}, \chi_{ZI}$
Coherent Z-Rotation
Detuning / Stark Shift
Recalibrate Pulse Frequency

Note: You can copy this entire block of text now and save it as a PDF or document on your end.
What other quantum computing topics can I help you with?

